In [ ]:
import numpy as np
# from scipy.integrate import cumtrapz
from fastdtw import fastdtw
from sklearn.cluster import DBSCAN

class GRBClassifier:
    def __init__(self, templates=None, dtw_threshold=12.0):
        """
        templates: A dictionary { 'classification_name': template_light_curve }
        dtw_threshold: Threshold distance below which an event is considered a GRB (12.0 for KL metric)
        """
        self.templates = templates if templates else {}
        self.dtw_threshold = dtw_threshold

    def normalize(self, data):
        """Rescale signal between 0 and 1 as per paper preprocessing[span_10](end_span)"""
        return (data - np.min(data)) / (np.max(data) - np.min(data) + 1e-9)

    def calculate_t90(self, light_curve, time_bins):
        """Calculates the T90 duration[span_11](end_span)"""
        cumulative_counts = np.cumsum(light_curve)
        total_counts = cumulative_counts[-1]
        
        # Find times for 5% and 95% of total fluence
        t5_idx = np.where(cumulative_counts >= 0.05 * total_counts)[0][0]
        t95_idx = np.where(cumulative_counts >= 0.95 * total_counts)[0][0]
        
        t90 = time_bins[t95_idx] - time_bins[t5_idx]
        return t90

    def kl_distance(self, x, y):
        """Symmetric Kullback-Leibler distance used in the paper[span_12](end_span)"""
        # Add epsilon to avoid log(0)
        x = x + 1e-6
        y = y + 1e-6
        return np.sum((x - y) * (np.log(x) - np.log(y)))

    def classify(self, input_data, time_bins):
        normalized_data = self.normalize(input_data)
        t90 = self.calculate_t90(normalized_data, time_bins)
        
        best_match = "Noise/Unknown"
        min_dist = float('inf')
        probabilities = {}

        # [span_13](start_span)Compare input to known templates using DTW[span_13](end_span)
        for name, template in self.templates.items():
            # Using standard Euclidean distance for 'fastdtw' default, 
            # [span_14](start_span)though paper suggests KL metric for better performance[span_14](end_span)
            distance, _ = fastdtw(normalized_data, template, dist=2) 
            
            # Simple inverse-distance weight for 'probability' representation
            probabilities[name] = 1.0 / (1.0 + distance)
            
            if distance < min_dist:
                min_dist = distance
                best_match = name

        # Final classification logic
        is_grb = min_dist < self.dtw_threshold
        grb_type = "Long GRB" if t90 >= 2.0 else "Short GRB"
        
        # Normalize weights to get pseudo-probabilities
        total_weight = sum(probabilities.values())
        if total_weight > 0:
            probabilities = {k: v / total_weight for k, v in probabilities.items()}

        return {
            "is_detected": is_grb,
            "morphology_match": best_match if is_grb else "Background Noise",
            "duration_class": grb_type if is_grb else "N/A",
            "t90_seconds": round(t90, 3),
            "dtw_distance": round(min_dist, 3),
            "morphology_probabilities": probabilities
        }

# --- Example Usage ---
# 1. Define dummy templates (for a real project, extract these from paper Fig 2 or AstroSat data)
templates = {
    "Cluster_1_Single_Peak": np.exp(-np.linspace(-2, 2, 50)**2),
    "Cluster_2_Multi_Peak": np.array([0.1, 0.5, 0.2, 0.8, 0.3, 0.1]) # Placeholder
}

classifier = GRBClassifier(templates=templates)

# 2. Simulate input data (e.g., a 10s light curve with 1s bins)
time_bins = np.arange(0, 10, 1)
sample_data = np.array([10, 12, 50, 100, 45, 15, 10, 11, 10, 9]) # A clear peak

# 3. Get Classification
results = classifier.classify(sample_data, time_bins)

print(f"Detection: {results['is_detected']}")
print(f"Type: {results['duration_class']} ({results['t90_seconds']}s)")
print(f"Morphology Probabilities: {results['morphology_probabilities']}")

<>:16: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
<>:21: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
<>:34: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
C:\Users\yashb\AppData\Local\Temp\ipykernel_19832\1106234567.py:16: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
  [span_10](start_span)
C:\Users\yashb\AppData\Local\Temp\ipykernel_19832\1106234567.py:21: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
  [span_11](start_span)
C:\Users\yashb\AppData\Local\Temp\ipykernel_19832\1106234567.py:34: SyntaxWarning: 'list' object is not callable; perhaps you missed a comma?
  [span_12](start_span)


NameError: name 'span_10' is not defined